In [57]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import pickle
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [58]:
df=pd.read_csv("loan.csv")

In [59]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [60]:
df.tail()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y
613,LP002990,Female,No,0,Graduate,Yes,4583,0.0,133.0,360.0,0.0,Semiurban,N


In [61]:
df.shape

(614, 13)

In [62]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [63]:
df=df.dropna()

In [64]:
df.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [65]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,480.000000,480.000000,480.000000,480.000000,480.000000
mean,5364.231250,1581.093583,144.735417,342.050000,0.854167
std,5668.251251,2617.692267,80.508164,65.212401,0.353307
min,150.000000,0.000000,9.000000,36.000000,0.000000
25%,2898.750000,0.000000,100.000000,360.000000,1.000000
50%,3859.000000,1084.500000,128.000000,360.000000,1.000000
75%,5852.500000,2253.250000,170.000000,360.000000,1.000000
max,81000.000000,33837.000000,600.000000,480.000000,1.000000


In [66]:
df.select_dtypes(include=['number']).corr()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
ApplicantIncome,1.000000,-0.112588,0.495310,-0.010838,-0.056152
CoapplicantIncome,-0.112588,1.000000,0.190740,-0.005775,-0.008692
LoanAmount,0.495310,0.190740,1.000000,0.050867,-0.040773
Loan_Amount_Term,-0.010838,-0.005775,0.050867,1.000000,0.032937
Credit_History,-0.056152,-0.008692,-0.040773,0.032937,1.000000


In [67]:
y=df["Loan_Status"]
x=df.drop(columns=["Loan_Status","Loan_ID"])

In [68]:
x

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban
5,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban
...,...,...,...,...,...,...,...,...,...,...,...
609,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural
610,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural
611,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban
612,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban


In [69]:
y

1      N
2      Y
3      Y
4      Y
5      Y
      ..
609    Y
610    Y
611    Y
612    Y
613    N
Name: Loan_Status, Length: 480, dtype: object

In [70]:
y.value_counts()

Loan_Status
Y    332
N    148
Name: count, dtype: int64

In [71]:
categorical_cols=x.select_dtypes(include=['object']).columns.tolist()
categorical_cols

['Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'Property_Area']

In [72]:
numerical_cols=x.select_dtypes(include=['int64','float64']).columns.tolist()
numerical_cols

['ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History']

In [73]:
preprocessor=ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown='ignore'),categorical_cols),
        ("num",StandardScaler(),numerical_cols)
    ]
)

In [74]:
from sklearn.pipeline import Pipeline

model=Pipeline([
    ("preprocess",preprocessor),
    ("rf",RandomForestClassifier())
])

In [75]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=55)

In [76]:
model.fit(X_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'Married',
                                                   'Dependents', 'Education',
                                                   'Self_Employed',
                                                   'Property_Area']),
                                                 ('num', StandardScaler(),
                                                  ['ApplicantIncome',
                                                   'CoapplicantIncome',
                                                   'LoanAmount',
                                                   'Loan_Amount_Term',
                                                   'Credit_History'])])),
                ('rf', RandomForestClassifier())])

In [77]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
y_pred=model.predict(X_test)

print("Model Evaluation on Test Set")
print("---------------------------------")

acc=accuracy_score(y_test,y_pred)
print(f"accuracy: {acc:4f}\n")


print(confusion_matrix(y_test,y_pred))
print("\n")

print("Classification Report:")
print(classification_report(y_test,y_pred))

Model Evaluation on Test Set
---------------------------------
accuracy: 0.875000

[[15  9]
 [ 3 69]]


Classification Report:
              precision    recall  f1-score   support

           N       0.83      0.62      0.71        24
           Y       0.88      0.96      0.92        72

    accuracy                           0.88        96
   macro avg       0.86      0.79      0.82        96
weighted avg       0.87      0.88      0.87        96



In [78]:
sample=pd.DataFrame([{
    "Gender":"Male",
    "Married":"yes",
    "Dependents":"1",
    "Education":"Graduate",
    "Self_Employed":"No",

    "ApplicantIncome":4500,
    "CoapplicantIncome":128,
    "LoanAmount":128,
    "Loan_Amount_Term":360,
    "Credit_History":1,
    "Property_Area":"Urban"
}])

In [79]:
print("Prediction:",model.predict(sample)[0])
print("Prediction:",model.predict_proba(sample)[0][1])

Prediction: Y
Prediction: 0.76


In [84]:
sample2=pd.DataFrame([{
    "Gender":"Male",
    "Married":"yes",
    "Dependents":"3+",
    "Education":"Graduate",
    "Self_Employed":"No",

    "ApplicantIncome":100,
    "CoapplicantIncome":128,
    "LoanAmount":400,
    "Loan_Amount_Term":360,
    "Credit_History":1,
    "Property_Area":"Urban"
}])

In [85]:
print("Prediction:",model.predict(sample2)[0])
print("Prediction:",model.predict_proba(sample2)[0][1])

Prediction: N
Prediction: 0.46


In [82]:
with open("loan_model.pkl","wb") as file:
    pickle.dump(model,file)